In [1]:
# Import required libraries
import os
import pandas as pd
from utils import log_standard_scale, peek
from pathlib import Path
RAW_DIR = Path("../data")

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_2019_UGA.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,2019_UGA.ID_208,0,ENSG00000000003,NaN,0.077926,Unknown
1,2019_UGA.ID_208,0,ENSG00000000005,NaN,0.000000,Unknown
2,2019_UGA.ID_208,0,ENSG00000000419,NaN,5.342548,Unknown
3,2019_UGA.ID_208,0,ENSG00000000457,NaN,9.799336,Unknown
4,2019_UGA.ID_208,0,ENSG00000000460,NaN,2.118530,Unknown


In [3]:
# Drop raw_count (entirely NaN) and material (entirely 'Unknown') before pivoting
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2019_UGA.ID_208,0,ENSG00000000003,0.077926
1,2019_UGA.ID_208,0,ENSG00000000005,0.000000
2,2019_UGA.ID_208,0,ENSG00000000419,5.342548
3,2019_UGA.ID_208,0,ENSG00000000457,9.799336
4,2019_UGA.ID_208,0,ENSG00000000460,2.118530


In [4]:
unique_genes_challenge = set(
    pd.read_csv(RAW_DIR / "challenge_transcriptomics.tsv", sep='\t')['ensembl_gene_id'].unique()
)
len(unique_genes_challenge)

54902

In [5]:
df['timepoint'].unique()

array([ 0,  3,  7, 28])

In [6]:
# Filter to only timepoints 0, 7. Timepoint 3 not in the challenge set
df = df[df['timepoint'].isin([0, 7])]
# Filter to only challenge genes
df = df[df["ensembl_gene_id"].isin(unique_genes_challenge)]

# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "d{timepoint}_{gene}" format
df_pivot.columns = [f'd{int(tp)}_{gene}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)

(275, 64125)


In [7]:
peek(df_pivot)

,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,d0_ENSG00000001167,d0_ENSG00000001460,d0_ENSG00000001461,d0_ENSG00000001497,d0_ENSG00000001561,d0_ENSG00000001617,d0_ENSG00000001626,d0_ENSG00000001629,d0_ENSG00000001630,d0_ENSG00000001631
0,2019_UGA.ID_001,0.149866,0.0,8.066946,8.457258,3.973795,95.386306,0.034801,13.583601,7.965179,7.187047,1.590390,13.973659,9.685137,3.203689,0.000000,0.0,10.685926,3.141505,8.707056
1,2019_UGA.ID_005,0.119232,0.0,3.209018,7.973900,2.521538,83.430649,0.110751,10.308613,12.190398,9.965058,1.533703,16.946756,10.774215,6.274083,0.029305,0.0,16.372074,2.892793,11.885934
2,2019_UGA.ID_008,0.096349,0.0,5.277265,7.289030,1.913696,138.440845,0.143194,10.557651,14.385306,5.401848,1.982958,9.914634,6.998915,2.006857,0.000000,0.0,4.988143,2.244099,6.075266
3,2019_UGA.ID_011,0.044666,0.0,3.733021,8.731205,3.491700,150.812198,0.074682,11.532723,14.860079,11.703771,1.149111,10.513365,6.511664,10.136052,0.000000,0.0,15.577613,2.566189,12.220556
4,2019_UGA.ID_014,0.000000,0.0,5.207440,9.466478,2.931771,129.608966,0.128051,15.970298,13.050625,10.539603,1.994948,15.887030,8.589723,10.012239,0.112938,0.0,14.760842,3.121698,10.802686


In [8]:
df_pivot = log_standard_scale(df_pivot)
peek(df_pivot)

,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,d0_ENSG00000001167,d0_ENSG00000001460,d0_ENSG00000001461,d0_ENSG00000001497,d0_ENSG00000001561,d0_ENSG00000001617,d0_ENSG00000001626,d0_ENSG00000001629,d0_ENSG00000001630,d0_ENSG00000001631
0,2019_UGA.ID_001,1.001284,-0.123083,0.651376,0.546033,1.491505,-0.622255,-1.007577,0.598130,0.109457,0.388386,0.512635,1.018868,0.923870,0.007177,-0.310983,-0.184023,0.478017,0.767752,0.390241
1,2019_UGA.ID_005,0.586238,-0.123083,-0.682261,0.408613,0.482631,-0.929775,-0.492218,0.042291,0.819701,0.915067,0.428503,1.455299,1.194103,0.816484,0.350294,-0.184023,1.078850,0.598956,0.949021
2,2019_UGA.ID_008,0.268739,-0.123083,0.012367,0.200666,-0.070990,0.235228,-0.282735,0.089897,1.102819,-0.055020,1.049170,0.256959,0.117698,-0.487343,-0.310983,-0.184023,-0.535201,0.102128,-0.233563
3,2019_UGA.ID_011,-0.473472,-0.123083,-0.478352,0.620830,1.193616,0.432613,-0.732424,0.266911,1.158720,1.180399,-0.197465,0.385646,-0.057288,1.445018,-0.310983,-0.184023,1.007912,0.360122,0.999589
4,2019_UGA.ID_014,-1.145122,-0.123083,-0.007073,0.811627,0.804596,0.083290,-0.379761,0.929377,0.935906,1.007133,1.064422,1.308638,0.622721,1.428517,2.138805,-0.184023,0.931346,0.754686,0.775816


In [9]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_parquet('../cleaned_data/transcriptomics_2019_UGA_cleaned.parquet', index=False)